# 📊 V-ChartText: Pipeline Tự Động Hóa Bước 3 & Bước 4 (Kiến Trúc 4 Bước)
### Ứng dụng VLM trong tự động trích xuất và sinh mô tả biểu đồ học tập tiếng Việt phục vụ hỗ trợ tiếp cận
**Repository:** [dangthihoangnganit6/V-ChartText](https://github.com/dangthihoangnganit6/V-ChartText)  
**Tài liệu tham chiếu:** Slide báo cáo giữa kì (`slide giữa kì_final.pdf` - Slide 3, 4, 6, 7, 9)  

---
### 🎯 Mục tiêu Notebook (`02_Batch_Step3_4.ipynb`):
1. **Bước 2 Nâng cao (Kiểm chứng lại số liệu):** Đối với các mẫu `validation_status != 'Hợp lệ'` từ Bước 1, gọi VLM đối chiếu lại ảnh gốc và thang đo trục tung (Y-axis). Gán cờ `"chưa chắc chắn"` trực tiếp vào giá trị mờ/nghi vấn, **tuyệt đối không đoán mò số liệu**.
2. **Bước 3 (Sinh mô tả 3 cấp độ - Text-Only VLM Mode):** Truyền bảng số liệu sạch vào VLM ở chế độ thuần văn bản (**không truyền ảnh**) để sinh 3 cấp độ mô tả tiếng Việt: `desc_5s` (nghe nhanh 5s), `desc_30s` (số liệu định lượng 30s), `desc_study` (diễn giải học tập không suy diễn nguyên nhân bên ngoài). Hậu kiểm số liệu bằng Regex (thước đo **ViChartFact**).
3. **Bước 4 (Chuẩn hóa đọc số tiếng Việt & Xuất Audio TTS):** Chuyển đổi ngữ âm tiếng Việt (`12,5%` $\rightarrow$ `mười hai phẩy năm phần trăm`, `1.200` $\rightarrow$ `một nghìn hai trăm`, `2010-2022` $\rightarrow$ `từ năm hai nghìn không trăm mười đến...`), tổng hợp giọng nói tự nhiên với `edge-tts` (`vi-VN-HoaiMyNeural`).
4. **Lưu trữ cuốn chiếu & Xuất Báo cáo:** Tự động resume nếu đứt runtime, xuất file CSV hoàn chỉnh 4 bước và file Excel tổng hợp nghiệm thu.

--- 
## ⚙️ Cell 1: Setup Môi Trường, Mount Google Drive & Nạp Mô Hình VLM
- Cài đặt các thư viện: `transformers==4.45.2`, `accelerate`, `qwen-vl-utils`, `edge-tts`, `pandas`, `openpyxl`, `tqdm`.
- Mount Google Drive để đọc dữ liệu Bước 1 (`vcharttext_extracted_results.csv`) và thư mục ảnh gốc.
- Nạp mô hình `Qwen/Qwen2-VL-2B-Instruct` (FP16, device_map='auto').

In [ ]:
# 1. Cài đặt các thư viện phụ thuộc
!pip install -q --upgrade pip
!pip install -q transformers==4.45.2 accelerate torchvision qwen-vl-utils edge-tts pandas openpyxl tqdm Pillow

import os
import sys
import re
import json
import unicodedata
import torch
import pandas as pd
from PIL import Image
from google.colab import drive
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info

# 2. Mount Google Drive
print("Đang kết nối Google Drive...")
drive.mount('/content/drive')

# 3. Thiết lập các đường dẫn dữ liệu bám sát yêu cầu đề tài
DRIVE_BASE = "/content/drive/MyDrive"
INPUT_CSV = f"{DRIVE_BASE}/vcharttext_extracted_results.csv"
IMAGE_DIR = f"{DRIVE_BASE}/Ảnh biểu đồ"
STEP3_CSV = f"{DRIVE_BASE}/vcharttext_step3_descriptions.csv"
FINAL_CSV = f"{DRIVE_BASE}/vcharttext_final_4steps.csv"
AUDIO_DIR = f"{DRIVE_BASE}/vcharttext_audio"
REPORT_EXCEL = f"{DRIVE_BASE}/vcharttext_summary_report.xlsx"
LOCAL_BACKUP_CSV = "/content/vcharttext_final_4steps.csv"

os.makedirs(AUDIO_DIR, exist_ok=True)

print(f"\n📂 File đầu vào Bước 1 & 2: {INPUT_CSV}")
print(f"📁 Thư mục ảnh gốc: {IMAGE_DIR}")
print(f"💾 File lưu Bước 3 (Mô tả): {STEP3_CSV}")
print(f"💾 File tổng hợp 4 Bước: {FINAL_CSV}")
print(f"🔊 Thư mục xuất Audio TTS: {AUDIO_DIR}")

# 4. Kiểm tra GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nThiết bị tính toán: {device.upper()}")
if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Card đồ họa: {gpu_name} ({vram_gb:.2f} GB VRAM)")

# 5. Nạp mô hình Qwen2-VL-2B-Instruct
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
print(f"\nĐang nạp mô hình {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28,
    trust_remote_code=True
)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
print("✅ Mô hình VLM đã sẵn sàng!")

--- 
## 🧠 Cell 2: Bước 2 Nâng Cao & Bước 3 - Sinh Mô Tả 3 Cấp Độ (Text-Only) & Hậu Kiểm Số Liệu

### Quy trình kỹ thuật:
1. **Bước 2 Nâng cao (Visual Re-verification):** Với các mẫu `validation_status != 'Hợp lệ'`, gọi VLM đối chiếu lại ảnh gốc và thang đo trục tung ($Y$-axis). Nếu số liệu mờ hoặc không xác minh được, gắn nhãn `"chưa chắc chắn"` trực tiếp vào ô giá trị đó (**tuyệt đối không bịa số**).
2. **Bước 3 (Mô tả đa cấp độ bám bảng số liệu - Slide 6 & 7):**
   - **Chế độ Text-Only:** Chỉ truyền bảng số liệu vào prompt của VLM (**không truyền ảnh**). Điều này loại bỏ hoàn toàn hiện tượng ảo giác thị giác (hallucination) và đảm bảo mọi câu chữ đều bám sát dữ liệu thực nghiệm.
   - **3 cấp độ mô tả chuẩn hóa:**
     - `desc_5s`: 1 câu tóm tắt nhanh (nghe 5 giây) để lướt nhanh loại biểu đồ và xu hướng tăng/giảm chính.
     - `desc_30s`: 3-4 câu số liệu định lượng (nghe 30 giây), nêu rõ mốc đỉnh, đáy, tốc độ tăng trưởng và đơn vị tính để phục vụ làm bài tập.
     - `desc_study`: Diễn giải học tập (1 đoạn văn ngắn), phân tích quan hệ giữa các đại lượng **dựa trên số liệu có trong bảng**; cấm tự suy diễn nguyên nhân kinh tế - xã hội bên ngoài.
3. **Hậu kiểm số liệu (ViChartFact Metric - Slide 9):** Dùng Regex quét mọi con số trong câu mô tả so với bảng số liệu. Nếu xuất hiện số lạ ngoài bảng $\rightarrow$ cảnh báo `Cảnh báo lệch số`.
4. **Resume Checkpoint:** Ghi cuốn chiếu từng dòng ra `vcharttext_step3_descriptions.csv`.

In [ ]:
import os
import re
import csv
import json
import unicodedata
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from qwen_vl_utils import process_vision_info

# ==============================================================================
# 1. LẬP CHỈ MỤC ẢNH GỐC ĐỂ PHỤC VỤ BƯỚC 2 NÂNG CAO
# ==============================================================================
print(f"🔍 Đang quét thư mục ảnh gốc: {IMAGE_DIR} ...")
image_path_map = {}
if os.path.exists(IMAGE_DIR):
    for root, _, files in os.walk(IMAGE_DIR):
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')):
                norm_key = unicodedata.normalize("NFC", f).strip().lower()
                image_path_map[norm_key] = os.path.join(root, f)
print(f"-> Đã lập chỉ mục {len(image_path_map)} ảnh gốc.")

# ==============================================================================
# 2. HÀM KIỂM CHỨNG LẠI SỐ LIỆU VỚI ẢNH GỐC (BƯỚC 2 NÂNG CAO)
# ==============================================================================
PROMPT_REVERIFY = """Bạn là chuyên gia kiểm định số liệu biểu đồ SGK tiếng Việt. Biểu đồ này đang bị nghi vấn chưa khớp giữa các trục và nhãn số.
Hãy quan sát kỹ hình ảnh biểu đồ, ĐẶC BIỆT LÀ THANG ĐO TRỤC TUNG (Y-axis) và các vạch chia:
Dữ liệu hiện có:
- Tiêu đề: {title}
- Đơn vị: {unit}
- Danh mục trục hoành (categories): {categories}
- Tên chuỗi (series_name): {series}
- Giá trị đọc được trước đó: {values}

YÊU CẦU KIỂM CHỨNG:
1. Đối chiếu lại từng vị trí cột/đường/miền với thang đo trục tung để xác minh số thực tế.
2. Giữ nguyên định dạng số gốc (số thập phân dùng dấu phẩy '12,5', hàng nghìn dùng dấu chấm '1.200').
3. NẾU BẤT KỲ GIÁ TRỊ NÀO BỊ MỜ, BỊ CHE KHUẤT HOẶC KHÔNG THỂ XÁC MINH CHẮC CHẮN TỪ HÌNH ẢNH: Hãy ghi đúng chữ 'chưa chắc chắn' vào đúng vị trí đó. TUYỆT ĐỐI KHÔNG ĐƯỢC ĐOÁN MÒ SỐ LIỆU.
4. Trả về DUY NHẤT một chuỗi JSON hợp lệ:
{
  "verified_values": "giá trị các chuỗi phân tách bằng | và ||",
  "verification_notes": "ghi chú ngắn gọn các mốc đã xác minh hoặc mốc nào chưa chắc chắn"
}
"""

def reverify_with_vlm(img_path, row_data):
    """Gọi VLM multimodal đối chiếu lại ảnh gốc để kiểm chứng số liệu bị nghi vấn."""
    prompt = PROMPT_REVERIFY.format(
        title=row_data.get("chart_title", ""),
        unit=row_data.get("unit", ""),
        categories=row_data.get("categories", ""),
        series=row_data.get("series_name", ""),
        values=row_data.get("values", "")
    )
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": prompt}
            ]
        }
    ]
    text_in = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text_in], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model.device)

    with torch.no_grad():
        gen_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    gen_trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen_ids)]
    raw = processor.batch_decode(gen_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    del inputs, gen_ids, gen_trimmed

    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        try:
            res = json.loads(match.group(0))
            return res.get("verified_values", row_data.get("values", "")), res.get("verification_notes", "Đã kiểm chứng lại qua ảnh")
        except Exception:
            pass
    return row_data.get("values", ""), "Không parse được kết quả kiểm chứng lại"

# ==============================================================================
# 3. HÀM SINH MÔ TẢ 3 CẤP ĐỘ Ở CHẾ ĐỘ THUẦN VĂN BẢN (TEXT-ONLY VLM)
# ==============================================================================
PROMPT_STEP3_TEXTONLY = """Bạn là trợ lý giáo dục chuyên tạo bản mô tả biểu đồ học tập tiếng Việt phục vụ người khiếm thị và hỗ trợ tiếp cận (SGK Lịch sử, Địa lí).
Dưới đây là BẢNG SỐ LIỆU ĐÃ ĐƯỢC KIỂM CHỨNG của một biểu đồ:
--------------------------------------------------
- Tiêu đề biểu đồ: {title}
- Loại biểu đồ: {chart_type}
- Đơn vị tính: {unit}
- Trục hoành / Danh mục: {x_label} ({categories})
- Tên chuỗi dữ liệu (chú giải): {series}
- Dữ liệu số chi tiết: {values}
--------------------------------------------------

Hãy đóng vai trò người đọc biểu đồ chuẩn mực, sinh DUY NHẤT một chuỗi JSON gồm 3 cấp độ mô tả:
{
  "desc_5s": "Mô tả 1 câu tóm tắt (nghe trong 5 giây): Nêu loại biểu đồ, đối tượng phản ánh và xu hướng tăng/giảm chính để người học nắm nhanh nội dung.",
  "desc_30s": "Mô tả định lượng chi tiết (nghe trong 30 giây, từ 3-4 câu): Cung cấp đầy đủ các mốc đỉnh (cao nhất), đáy (thấp nhất), tốc độ tăng/giảm và các giá trị cụ thể kèm đơn vị tính để phục vụ làm bài tập.",
  "desc_study": "Diễn giải học tập (1 đoạn văn súc tích từ 4-6 câu): Phân tích mối quan hệ giữa các đại lượng, nhận xét học tập rút ra từ số liệu. QUY TẮC SỐNG CÒN: Chỉ diễn giải dựa trên dữ liệu có trong bảng, TUYỆT ĐỐI KHÔNG TỰ SUY DIỄN các nguyên nhân kinh tế - xã hội bên ngoài không có căn cứ từ biểu đồ."
}

QUY TẮC BẮT BUỘC:
1. BÁM SÁT 100% BẢNG DỮ LIỆU: Mọi con số trong bài mô tả phải khớp chính xác với bảng dữ liệu ở trên.
2. XỬ LÝ SỐ LIỆU CHƯA CHẮC CHẮN: Nếu trong bảng có giá trị ghi 'chưa chắc chắn', câu mô tả phải nêu rõ mốc đó 'chưa thể xác định chắc chắn từ biểu đồ', không được bịa số.
3. NGÔN NGỮ: Viết bằng tiếng Việt tự nhiên, chuẩn mực sư phạm, câu từ rõ ràng, mạch lạc.
4. ĐỊNH DẠNG ĐẦU RA: Chỉ trả về chuỗi JSON thuần (bắt đầu bằng { và kết thúc bằng }), không có lời mở đầu hay kết thúc.
"""

def generate_descriptions_textonly(table_row):
    """Gọi Qwen2-VL ở chế độ Text-Only (không truyền ảnh) để sinh 3 cấp độ mô tả."""
    prompt = PROMPT_STEP3_TEXTONLY.format(
        title=table_row.get("chart_title", "Biểu đồ không có tên"),
        chart_type=table_row.get("chart_type", "biểu đồ"),
        unit=table_row.get("unit", "đơn vị"),
        x_label=table_row.get("x_axis_label", "Thời gian/Đối tượng"),
        categories=table_row.get("categories", ""),
        series=table_row.get("series_name", ""),
        values=table_row.get("values", "")
    )

    # Định dạng đầu vào thuần văn bản (KHÔNG CÓ KHỐI IMAGE)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt}
            ]
        }
    ]
    text_in = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text_in], padding=True, return_tensors="pt").to(model.device)

    with torch.no_grad():
        gen_ids = model.generate(**inputs, max_new_tokens=1024, do_sample=False)
    gen_trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen_ids)]
    raw = processor.batch_decode(gen_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    del inputs, gen_ids, gen_trimmed

    # Parse JSON an toàn
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        try:
            data = json.loads(match.group(0))
            return data.get("desc_5s", ""), data.get("desc_30s", ""), data.get("desc_study", ""), None
        except Exception as e:
            return "", "", "", f"Lỗi parse JSON mô tả ({e})"
    return "", "", "", "Không tìm thấy JSON trong output mô tả"

# ==============================================================================
# 4. HÀM HẬU KIỂM SỐ LIỆU BẰNG REGEX (VICHARTFACT METRIC - SLIDE 9)
# ==============================================================================
def fact_check_text_against_table(desc_text, table_values_str, categories_str):
    """
    Hậu kiểm số liệu: Trích xuất các số trong câu mô tả và đối chiếu với bảng số liệu.
    """
    if not desc_text:
        return "Chưa có văn bản", "Nội dung rỗng"

    # Thu thập các token số trong bảng (chuẩn hóa cả dạng chấm và phẩy)
    table_tokens = set()
    for chunk in f"{table_values_str} | {categories_str}".replace("||", "|").split("|"):
        clean_chunk = chunk.strip().replace("%", "")
        if clean_chunk:
            table_tokens.add(clean_chunk)
            table_tokens.add(clean_chunk.replace(",", "."))
            table_tokens.add(clean_chunk.replace(".", ","))

    # Tìm các con số xuất hiện trong đoạn văn (ví dụ: 12,5, 1.200, 2020,...)
    found_numbers = re.findall(r"\b\d+(?:[.,]\d+)?\b", desc_text)
    unmatched = []
    for num in found_numbers:
        # Bỏ qua các số nhỏ thể hiện số câu, số cấp độ như 5, 30 nếu là tên nhãn
        if num in ["5", "30", "1", "2", "3", "4"] and num not in table_tokens:
            continue
        num_dot = num.replace(",", ".")
        num_comma = num.replace(".", ",")
        if num not in table_tokens and num_dot not in table_tokens and num_comma not in table_tokens:
            unmatched.append(num)

    if not unmatched:
        return "Đạt chuẩn (100% khớp bảng)", "Tất cả các số liệu đều có trong bảng"
    else:
        return "Cảnh báo lệch số", f"Số có thể bị tính toán hoặc ngoài bảng: {', '.join(unmatched[:4])}"

# ==============================================================================
# 5. VÒNG LẶP THỰC THI BƯỚC 2 NÂNG CAO + BƯỚC 3 VỚI RESUME CHECKPOINT
# ==============================================================================
if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(f"Không tìm thấy file đầu vào: {INPUT_CSV}. Hãy chạy notebook Bước 1 trước!")

df_input = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
print(f"\n📊 Đã nạp {len(df_input)} dòng từ {INPUT_CSV}")

STEP3_FIELDNAMES = [
    "STT", "image_name", "chart_title", "chart_type", "unit", "x_axis_label",
    "categories", "series_name", "values", "validation_status",
    "desc_5s", "desc_30s", "desc_study", "fact_check_status", "fact_check_notes"
]

# Kiểm tra resume
processed_step3_images = set()
if os.path.exists(STEP3_CSV) and os.path.getsize(STEP3_CSV) > 0:
    try:
        df_s3 = pd.read_csv(STEP3_CSV, encoding="utf-8-sig")
        if "image_name" in df_s3.columns:
            processed_step3_images = set(df_s3["image_name"].dropna().astype(str).str.strip())
            print(f"🔄 Checkpoint: Đã xử lý {len(processed_step3_images)} ảnh ở Bước 3. Tiếp tục với các ảnh còn lại!")
    except Exception as e:
        print(f"⚠️ Không đọc được checkpoint cũ: {e}")

if not os.path.exists(STEP3_CSV) or os.path.getsize(STEP3_CSV) == 0:
    with open(STEP3_CSV, mode="w", encoding="utf-8-sig", newline="") as f_init:
        writer = csv.DictWriter(f_init, fieldnames=STEP3_FIELDNAMES)
        writer.writeheader()

rows_to_process = df_input[~df_input["image_name"].astype(str).str.strip().isin(processed_step3_images)]
print(f"🚀 Bắt đầu xử lý {len(rows_to_process)} biểu đồ cho Bước 2 nâng cao & Bước 3...\n")

for idx, row in tqdm(rows_to_process.iterrows(), total=len(rows_to_process), desc="Tiến độ Bước 3"):
    img_name = str(row.get("image_name", "")).strip()
    norm_key = unicodedata.normalize("NFC", img_name).lower()
    val_status = str(row.get("validation_status", "")).strip()
    curr_values = str(row.get("values", "")).strip()

    # --- 1. Bước 2 Nâng cao: Kiểm chứng lại số liệu nếu status != 'Hợp lệ' ---
    if val_status != "Hợp lệ" and norm_key in image_path_map:
        img_path = image_path_map[norm_key]
        try:
            reverified_val, reverify_note = reverify_with_vlm(img_path, row.to_dict())
            curr_values = reverified_val
            val_status = "Đã kiểm chứng lại"
        except Exception as e:
            val_status = "Chưa chắc chắn"

    # Cập nhật row dict với values đã kiểm chứng
    row_dict = row.to_dict()
    row_dict["values"] = curr_values
    row_dict["validation_status"] = val_status

    # --- 2. Bước 3: Sinh mô tả 3 cấp độ (Text-Only Mode) ---
    try:
        desc_5s, desc_30s, desc_study, desc_err = generate_descriptions_textonly(row_dict)
    except Exception as e:
        desc_5s, desc_30s, desc_study, desc_err = "", "", "", str(e)

    # --- 3. Hậu kiểm số liệu bằng Regex ---
    fc_status, fc_note = fact_check_text_against_table(
        f"{desc_5s} {desc_30s}",
        curr_values,
        str(row.get("categories", ""))
    )

    out_record = {
        "STT": row.get("STT", idx + 1),
        "image_name": img_name,
        "chart_title": row.get("chart_title", ""),
        "chart_type": row.get("chart_type", ""),
        "unit": row.get("unit", ""),
        "x_axis_label": row.get("x_axis_label", ""),
        "categories": row.get("categories", ""),
        "series_name": row.get("series_name", ""),
        "values": curr_values,
        "validation_status": val_status,
        "desc_5s": desc_5s,
        "desc_30s": desc_30s,
        "desc_study": desc_study,
        "fact_check_status": fc_status,
        "fact_check_notes": fc_note
    }

    # Ghi cuốn chiếu ngay vào file CSV trên Drive
    with open(STEP3_CSV, mode="a", encoding="utf-8-sig", newline="") as f_out:
        writer = csv.DictWriter(f_out, fieldnames=STEP3_FIELDNAMES)
        writer.writerow(out_record)
        f_out.flush()
        os.fsync(f_out.fileno())

    # In log tóm tắt
    icon = "✅" if fc_status.startswith("Đạt") else "⚠️"
    tqdm.write(f"{icon} [{img_name}] Status: {val_status} | FactCheck: {fc_status}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "="*60)
print(f"🎉 HOÀN THÀNH BƯỚC 2 NÂNG CAO VÀ BƯỚC 3!")
print(f"- Kết quả đã lưu an toàn tại: {STEP3_CSV}")
print("="*60)

--- 
## 🔊 Cell 3: Bước 4 - Chuẩn Hóa Đọc Số Tiếng Việt & Xuất Audio TTS Hỗ Trợ Tiếp Cận

### Cơ chế xử lý ngữ âm & TTS (Slide 6 & 7):
1. **Hàm `normalize_vn_tts(text)`:**
   - Chuyển số thập phân: `12,5%` $\rightarrow$ `mười hai phẩy năm phần trăm`.
   - Chuyển số hàng nghìn có dấu chấm: `1.200` $\rightarrow$ `một nghìn hai trăm`, `15.400` $\rightarrow$ `mười lăm nghìn bốn trăm`.
   - Chuyển khoảng năm SGK: `2010-2022` $\rightarrow$ `từ năm hai nghìn không trăm mười đến năm hai nghìn không trăm hai mươi hai`.
   - Chuyển các ký hiệu đơn vị đo: `ha` $\rightarrow$ `héc-ta`, `km2` $\rightarrow$ `ki-lô-mét vuông`, `USD` $\rightarrow$ `đô la Mỹ`.
2. **Xuất Audio đa cấp độ (`edge-tts`):**
   - Sử dụng giọng đọc chuẩn `vi-VN-HoaiMyNeural`.
   - Tạo 2 file audio độc lập cho mỗi biểu đồ: `{image_name}_5s.mp3` (dành cho lướt nhanh) và `{image_name}_30s.mp3` (dành cho làm bài tập).
   - Lưu kết quả tổng hợp 4 bước ra `/content/drive/MyDrive/vcharttext_final_4steps.csv`.

In [ ]:
import os
import re
import csv
import asyncio
import concurrent.futures
import edge_tts
import pandas as pd
from tqdm import tqdm

# ==============================================================================
# 1. BỘ CHUYỂN ĐỔI SỐ NGUYÊN THÀNH TỪ TIẾNG VIỆT CHUẨN SƯ PHẠM
# ==============================================================================
def int_to_vietnamese_words(n):
    """Chuyển số nguyên n thành chuỗi phát âm tiếng Việt."""
    if n == 0:
        return "không"
    digits = ["không", "một", "hai", "ba", "bốn", "năm", "sáu", "bảy", "tám", "chín"]
    units = ["", "nghìn", "triệu", "tỷ", "nghìn tỷ", "triệu tỷ"]

    def read_three_digits(num, is_highest=False):
        h = num // 100
        t = (num % 100) // 10
        u = num % 10
        res = []
        if h > 0 or not is_highest:
            res.append(digits[h] + " trăm")
        if t > 1:
            res.append(digits[t] + " mươi")
            if u == 1:
                res.append("mốt")
            elif u == 4:
                res.append("tư")
            elif u == 5:
                res.append("lăm")
            elif u > 0:
                res.append(digits[u])
        elif t == 1:
            res.append("mười")
            if u == 5:
                res.append("lăm")
            elif u > 0:
                res.append(digits[u])
        elif t == 0:
            if u > 0:
                if not is_highest or h > 0:
                    res.append("linh " + digits[u])
                else:
                    res.append(digits[u])
        return " ".join(res)

    groups = []
    temp = n
    while temp > 0:
        groups.append(temp % 1000)
        temp //= 1000

    parts = []
    for i in range(len(groups) - 1, -1, -1):
        g = groups[i]
        if g > 0:
            is_highest = (i == len(groups) - 1)
            parts.append(read_three_digits(g, is_highest) + (" " + units[i] if units[i] else ""))
    return " ".join(parts).strip()

# ==============================================================================
# 2. HÀM CHUẨN HÓA VĂN BẢN TRƯỚC KHI ĐỌC TTS (NORMALIZE_VN_TTS)
# ==============================================================================
def normalize_vn_tts(text):
    """
    Chuẩn hóa văn bản số liệu theo cách phát âm tiếng Việt tự nhiên:
    - '12,5%' -> 'mười hai phẩy năm phần trăm'
    - '1.200' -> 'một nghìn hai trăm'
    - '2010-2022' -> 'từ năm hai nghìn không trăm mười đến năm hai nghìn không trăm hai mươi hai'
    """
    if not text:
        return ""

    digits_map = {
        "0": "không", "1": "một", "2": "hai", "3": "ba", "4": "bốn",
        "5": "năm", "6": "sáu", "7": "bảy", "8": "tám", "9": "chín"
    }

    # 1. Khoảng năm SGK: 2010 - 2022 hoặc 2010-2022
    def rep_year_range(match):
        y1 = int(match.group(1))
        y2 = int(match.group(2))
        return f"từ năm {int_to_vietnamese_words(y1)} đến năm {int_to_vietnamese_words(y2)}"
    text = re.sub(r"(?:từ\s+)?(?:năm\s+)?\b(19\d{2}|20\d{2})\s*[-–—]\s*(19\d{2}|20\d{2})\b", rep_year_range, text, flags=re.IGNORECASE)

    # 2. Năm đơn lẻ: năm 2020
    def rep_single_year(match):
        y = int(match.group(1))
        return f"năm {int_to_vietnamese_words(y)}"
    text = re.sub(r"\bnăm\s+(19\d{2}|20\d{2})\b", rep_single_year, text, flags=re.IGNORECASE)

    # 3. Ký hiệu phần trăm
    text = re.sub(r"%", " phần trăm", text)

    # 4. Số thập phân tiếng Việt dùng dấu phẩy (vd: 12,5)
    def rep_decimal(match):
        int_val = int(match.group(1))
        dec_str = match.group(2)
        dec_words = " ".join([digits_map.get(d, d) for d in dec_str])
        return f"{int_to_vietnamese_words(int_val)} phẩy {dec_words}"
    text = re.sub(r"\b(\d+),(\d+)\b", rep_decimal, text)

    # 5. Số hàng nghìn có dấu chấm (vd: 1.200, 15.400)
    def rep_thousands(match):
        raw_num = match.group(1).replace(".", "")
        return int_to_vietnamese_words(int(raw_num))
    text = re.sub(r"\b(\d{1,3}(?:\.\d{3})+)\b", rep_thousands, text)

    # 6. Các số nguyên độc lập còn lại
    def rep_integer(match):
        val = int(match.group(1))
        return int_to_vietnamese_words(val)
    text = re.sub(r"\b(\d+)\b", rep_integer, text)

    # 7. Các đơn vị đo thông dụng trong SGK
    unit_replacements = [
        (r"\bha\b", "héc-ta"),
        (r"\bkm2\b|\bkm²\b", "ki-lô-mét vuông"),
        (r"\bm2\b|\bm²\b", "mét vuông"),
        (r"\bUSD/người\b", "đô la Mỹ trên người"),
        (r"\bUSD\b", "đô la Mỹ"),
        (r"\btỉ\b", "tỷ")
    ]
    for pattern, rep in unit_replacements:
        text = re.sub(pattern, rep, text, flags=re.IGNORECASE)

    return re.sub(r"\s+", " ", text).strip()

# ==============================================================================
# 3. HÀM TỔNG HỢP GIỌNG NÓI AN TOÀN VỚI EDGE-TTS (vi-VN-HoaiMyNeural)
# ==============================================================================
def generate_speech_audio(text, output_mp3_path, voice="vi-VN-HoaiMyNeural"):
    """Xuất file mp3 từ văn bản tiếng Việt sử dụng edge-tts, chạy an toàn trong Colab."""
    if not text or not text.strip():
        return False

    norm_text = normalize_vn_tts(text)

    async def _speak():
        comm = edge_tts.Communicate(norm_text, voice)
        await comm.save(output_mp3_path)

    def _run():
        asyncio.run(_speak())

    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
            executor.submit(_run).result(timeout=30)
        return True
    except Exception as e:
        print(f"⚠️ Lỗi TTS cho {output_mp3_path}: {e}")
        return False

# ==============================================================================
# 4. TIẾN HÀNH XUẤT AUDIO VÀ LƯU BẢNG TỔNG HỢP 4 BƯỚC
# ==============================================================================
if not os.path.exists(STEP3_CSV):
    raise FileNotFoundError(f"Chưa tìm thấy file {STEP3_CSV}. Vui lòng chạy Cell 2 trước!")

df_step3 = pd.read_csv(STEP3_CSV, encoding="utf-8-sig")
print(f"\n📊 Đã nạp {len(df_step3)} dòng mô tả từ Bước 3.")

FINAL_FIELDNAMES = [
    "STT", "image_name", "chart_title", "chart_type", "unit", "x_axis_label",
    "categories", "series_name", "values", "validation_status",
    "desc_5s", "desc_30s", "desc_study", "fact_check_status", "fact_check_notes",
    "audio_5s_path", "audio_30s_path"
]

# Resume Checkpoint cho file kết quả cuối cùng
processed_final_images = set()
if os.path.exists(FINAL_CSV) and os.path.getsize(FINAL_CSV) > 0:
    try:
        df_final_old = pd.read_csv(FINAL_CSV, encoding="utf-8-sig")
        if "image_name" in df_final_old.columns:
            processed_final_images = set(df_final_old["image_name"].dropna().astype(str).str.strip())
            print(f"🔄 Checkpoint: Đã hoàn thành TTS cho {len(processed_final_images)} biểu đồ.")
    except Exception:
        pass

if not os.path.exists(FINAL_CSV) or os.path.getsize(FINAL_CSV) == 0:
    with open(FINAL_CSV, mode="w", encoding="utf-8-sig", newline="") as f_init:
        writer = csv.DictWriter(f_init, fieldnames=FINAL_FIELDNAMES)
        writer.writeheader()

rows_for_tts = df_step3[~df_step3["image_name"].astype(str).str.strip().isin(processed_final_images)]
print(f"🔊 Bắt đầu tạo Audio TTS cho {len(rows_for_tts)} biểu đồ...\n")

audio_count = 0
for idx, row in tqdm(rows_for_tts.iterrows(), total=len(rows_for_tts), desc="Tạo Audio TTS"):
    img_name = str(row.get("image_name", "")).strip()
    clean_base = os.path.splitext(img_name)[0].replace(" ", "_")

    audio_5s_file = f"{AUDIO_DIR}/{clean_base}_5s.mp3"
    audio_30s_file = f"{AUDIO_DIR}/{clean_base}_30s.mp3"

    # 1. Tạo audio 5s
    desc_5s_text = str(row.get("desc_5s", "")).strip()
    if desc_5s_text and (not os.path.exists(audio_5s_file) or os.path.getsize(audio_5s_file) == 0):
        ok_5s = generate_speech_audio(desc_5s_text, audio_5s_file)
        if ok_5s: audio_count += 1

    # 2. Tạo audio 30s
    desc_30s_text = str(row.get("desc_30s", "")).strip()
    if desc_30s_text and (not os.path.exists(audio_30s_file) or os.path.getsize(audio_30s_file) == 0):
        ok_30s = generate_speech_audio(desc_30s_text, audio_30s_file)
        if ok_30s: audio_count += 1

    # Bản ghi kết quả đầy đủ 4 bước
    final_record = row.to_dict()
    final_record["audio_5s_path"] = audio_5s_file if os.path.exists(audio_5s_file) else ""
    final_record["audio_30s_path"] = audio_30s_file if os.path.exists(audio_30s_file) else ""

    # Ghi cuốn chiếu ra file Drive và local backup
    with open(FINAL_CSV, mode="a", encoding="utf-8-sig", newline="") as f_out:
        writer = csv.DictWriter(f_out, fieldnames=FINAL_FIELDNAMES)
        writer.writerow(final_record)
        f_out.flush()
        os.fsync(f_out.fileno())

    try:
        with open(LOCAL_BACKUP_CSV, mode="a", encoding="utf-8-sig", newline="") as f_loc:
            writer_loc = csv.DictWriter(f_loc, fieldnames=FINAL_FIELDNAMES)
            writer_loc.writerow(final_record)
            f_loc.flush()
    except Exception:
        pass

print("\n" + "="*60)
print(f"🎉 HOÀN TẤT TOÀN DIỆN BƯỚC 4!")
print(f"- Tổng số file Audio mới đã xuất: {audio_count}")
print(f"- Kết quả 4 bước đã lưu tại: {FINAL_CSV}")
print("="*60)

--- 
## 📊 Cell 4: Thống Kê Nghiệm Thu, Xuất Excel & Tải Kết Quả Về Máy

Cell này thực hiện:
1. Tổng hợp các chỉ số thực nghiệm theo tiêu chí Slide 8 & 9 (tỷ lệ đạt chuẩn **ViChartFact**, phân bố loại biểu đồ, số lượng audio).
2. Xuất file Excel tổng hợp đa trang (`vcharttext_summary_report.xlsx`).
3. Tải trực tiếp file Excel và CSV kết quả về máy tính để đưa vào phụ lục báo cáo giữa kì / cuối kì.

In [ ]:
import os
import pandas as pd
from google.colab import files

target_csv = FINAL_CSV if os.path.exists(FINAL_CSV) else STEP3_CSV

if os.path.exists(target_csv) and os.path.getsize(target_csv) > 0:
    df_report = pd.read_csv(target_csv, encoding="utf-8-sig")
    total_charts = len(df_report)

    print("="*65)
    print(f"📈 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ THỰC NGHIỆM ĐỀ TÀI V-CHARTTEXT")
    print(f"   Tổng số mẫu biểu đồ đã xử lý qua 4 bước: {total_charts}")
    print("="*65)

    # 1. Thống kê trạng thái kiểm chứng số liệu (Bước 2)
    print("\n1. Trạng thái kiểm chứng dữ liệu bảng (Validation Status):")
    print(df_report["validation_status"].value_counts())

    # 2. Thống kê tỷ lệ hậu kiểm số liệu (ViChartFact Metric - Slide 9)
    if "fact_check_status" in df_report.columns:
        print("\n2. Độ trung thực số liệu mô tả (ViChartFact Fact-Checking):")
        fc_counts = df_report["fact_check_status"].value_counts()
        print(fc_counts)
        pass_count = fc_counts.get("Đạt chuẩn (100% khớp bảng)", 0)
        print(f"-> TỶ LỆ MÔ TẢ KHỚP 100% SỐ LIỆU BẢNG: {pass_count / total_charts * 100:.2f}%")

    # 3. Phân bố loại biểu đồ theo thực tế (Đối chiếu Slide 8)
    print("\n3. Phân bố các dạng biểu đồ trích xuất:")
    print(df_report["chart_type"].value_counts())

    # 4. Thống kê số lượng file Audio
    audio_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith(".mp3")] if os.path.exists(AUDIO_DIR) else []
    print(f"\n4. Số lượng file Audio hỗ trợ tiếp cận đã tạo: {len(audio_files)} files trong '{AUDIO_DIR}'")

    # 5. Xuất file Excel nhiều sheet phục vụ làm báo cáo nghiệm thu
    print(f"\n📄 Đang tạo file Excel báo cáo: {REPORT_EXCEL} ...")
    with pd.ExcelWriter(REPORT_EXCEL, engine="openpyxl") as writer:
        # Sheet 1: Toàn bộ dữ liệu 4 bước
        df_report.to_excel(writer, sheet_name="Du_Lieu_4_Buoc", index=False)

        # Sheet 2: Thống kê loại biểu đồ
        df_chart_type = df_report["chart_type"].value_counts().reset_index()
        df_chart_type.columns = ["Loai_Bieu_Do", "So_Luong"]
        df_chart_type.to_excel(writer, sheet_name="Thong_Ke_Bieu_Do", index=False)

        # Sheet 3: Danh sách các mẫu cần kiểm tra lại
        warning_df = df_report[df_report["validation_status"].isin(["Cần kiểm tra", "Chưa chắc chắn", "Lỗi trích xuất"])]
        if not warning_df.empty:
            warning_df.to_excel(writer, sheet_name="Mau_Can_Kiem_Tra", index=False)

    print(f"✅ Đã tạo file Excel tổng hợp thành công tại: {REPORT_EXCEL}")

    # 6. Tải file về máy tính cá nhân
    print("\n📥 Đang kích hoạt tải file Excel và CSV kết quả...")
    try:
        files.download(REPORT_EXCEL)
        files.download(target_csv)
        print("✅ Trình duyệt đang tải 2 file báo cáo về máy tính của bạn.")
    except Exception as e:
        print(f"ℹ️ Nếu tải tự động bị chặn, bạn có thể tải thủ công tại:\n- {REPORT_EXCEL}\n- {target_csv}")

    # 7. Hiển thị mẫu 3 biểu đồ đầu tiên đã hoàn thành 4 bước
    print("\n🔍 MẪU KẾT QUẢ ĐẦU RA 4 BƯỚC TIÊU BIỂU:")
    cols_show = ["image_name", "chart_type", "desc_5s", "desc_30s", "fact_check_status"]
    display(df_report[cols_show].head(3))

else:
    print(f"⚠️ Không tìm thấy file dữ liệu kết quả tại {target_csv}")